# Zadanie 2.4 / 2.7: generator danych do Zadań 3–8

Generujemy powtarzalny zbiór trajektorii PDE (issue #15) w `data/processed/` przez `data.make_dataset`
i `data.sample_scenarios` ze stałym ziarnem `SEED = 2026`.

- **Zbiór treningowy (`data/processed/train/`):** 80 symulacji z regularnymi harmonogramami (6–12 frakcji, odstęp 0,35–0,55)
  oraz planami wiązki $r \in [3, 16]$, $s \in [0, 8]$.
- **Zbiór testowy (`data/processed/test/`):** 20 symulacji z harmonogramami spoza treningu (3–5 rzadkich frakcji,
  14–18 gęstych frakcji lub dwa bloki z przerwą) i planami wiązki spoza prostokąta treningowego ($r < 3$, $r > 16$ lub $s > 8$).

Pliki `.npz` nie trafiają do gita — odtwarza je `uv run reproduce --only zad2_04_dane`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import DATA_PROCESSED, FIGURES, TABLES, seed_everything
from offtarget.data import load_dataset, make_dataset, sample_scenarios
from offtarget.model import grid
from offtarget.observables import min_mass, time_to_regrowth

rng = seed_everything()

x, dx = grid(80.0, 400)
t = np.round(np.linspace(0.0, 10.0, 201), 10)

t0 = time.perf_counter()
p_train, s_train = sample_scenarios(80, rng, split="train")
p_test, s_test = sample_scenarios(20, rng, split="test")

train_paths = make_dataset(p_train, s_train, x, t, out_dir=DATA_PROCESSED / "train", prefix="train", dt_max=0.05)
test_paths = make_dataset(p_test, s_test, x, t, out_dir=DATA_PROCESSED / "test", prefix="test", dt_max=0.05)
elapsed = time.perf_counter() - t0
print(f"Wygenerowano {len(train_paths)} train + {len(test_paths)} test w {elapsed:.1f} s")

## Przegląd wygenerowanych zbiorów

In [ ]:
rows = []
for split, folder in [("train", DATA_PROCESSED / "train"), ("test", DATA_PROCESSED / "test")]:
    for idx, run in enumerate(load_dataset(folder)):
        p, sched, M = run["params"], run["schedule"], run["M"]
        m_min, _ = min_mass(t, M)
        diffs = np.diff(sched.times)
        rows.append({
            "split": split,
            "id": idx,
            "D": p.D,
            "rho": p.rho,
            "beta": p.beta,
            "r": p.r,
            "s": p.s,
            "n_frakcji": len(sched.times),
            "min_odstep": float(diffs.min()),
            "max_odstep": float(diffs.max()),
            "M_min / M(0)": m_min / M[0],
            "czas_odrostu": time_to_regrowth(t, M),
            "M(10) / M(0)": float(M[-1] / M[0]),
        })
meta = pd.DataFrame(rows)
meta.to_csv(TABLES / "zad2_04_dane.csv", index=False)
meta.groupby("split")[["D", "rho", "beta", "r", "s", "n_frakcji", "M_min / M(0)", "M(10) / M(0)"]].agg(["min", "median", "max"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for split, col, mk in [("train", "C0", "o"), ("test", "C3", "^")]:
    sub = meta[meta["split"] == split]
    axes[0].scatter(sub["r"], sub["s"], c=col, marker=mk, alpha=0.8, label=split)
    axes[1].scatter(sub["n_frakcji"], sub["min_odstep"], c=col, marker=mk, alpha=0.8, label=split)
axes[0].add_patch(plt.Rectangle((3.0, 0.0), 13.0, 8.0, fill=False, ls="--", ec="0.4"))
axes[0].set(xlabel="r", ylabel="s", title="Plany wiązki: train vs test")
axes[0].legend()
axes[1].set(xlabel="liczba frakcji", ylabel="minimalny odstęp frakcji", title="Harmonogramy: train vs test")
axes[1].legend()

for split, folder, col in [("train", DATA_PROCESSED / "train", "C0"), ("test", DATA_PROCESSED / "test", "C3")]:
    for k, run in enumerate(load_dataset(folder)[:15]):
        M = run["M"]
        axes[2].semilogy(t, np.maximum(M / M[0], 1e-5), color=col, alpha=0.35, lw=1.2,
                         label=split if k == 0 else None)
axes[2].set(xlabel="czas t", ylabel="M(t) / M(0)", title="Przykładowe krzywe M(t)")
axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURES / "zad2_04_dane.png", dpi=150)

## Podsumowanie

- Generator zapisuje 80 trajektorii treningowych (`data/processed/train/train_0000.npz`–`train_0079.npz`)
  i 20 testowych (`data/processed/test/test_0000.npz`–`test_0019.npz`) z polami `x`, `t`, `u`, `M`, `params` i `schedule`.
- Podział trening/test jest ściśle rozłączny zarówno w płaszczyźnie planu wiązki $(r, s)$, jak i w harmonogramach frakcji
  (co weryfikuje test `tests/test_data.py`).
- Całość odtwarza się deterministycznie w kilka sekund poleceniem `uv run reproduce --only zad2_04_dane`.